# Demo — Making Document Extraction Worth Automating

Runs on a CPU in about 5 minutes; nothing is trained here. Finished experiments are read from the repository's `results` branch.

| Part | Shows |
|---|---|
| 1 | The problem: test sets reuse training layouts, measured with calibrated template clustering |
| 2 | Extraction results: models, methods, familiar vs unfamiliar layouts (Δ) |
| 3 | Routing: which documents can skip human review |
| 4 | One document, end to end: fields, arithmetic check, risk score, decision |
| 5 | How the experiments ran: an automated job queue |

In [ ]:
from pathlib import Path
ROOT = Path('/content/doc-extraction')
if not ROOT.exists():
    !git clone -q https://github.com/AviDip/doc-extraction {ROOT}
%cd {ROOT}
!git pull -q && git fetch -q origin results && git archive origin/results runs logs data status.json | tar -x
!pip freeze | grep -iE '^(numpy|pandas|torch|scipy)==' > /tmp/pins.txt && pip install -q -r requirements.txt -c /tmp/pins.txt
!python -m docext.prepare cord --out data > /dev/null && python -m docext.prepare cord-disjoint --out data --threshold 0.7 > /dev/null
import json, pandas as pd
from IPython.display import Image, display
from docext.demo import show, pick
RUN = 'runs/cord_layoutlmv3_j0.0_k0.0_a0.0_c0.0_s0'

## 1. The problem: familiar layouts inflate test scores

A receipt's template is identified by its header (shop name, address) and key wording. The clustering threshold is calibrated on WildReceipt, whose authors guarantee unseen test shops: the right threshold finds almost none of them "seen".

In [ ]:
cal = {name: pd.DataFrame(json.loads(Path(f'data/{name}/calibration.json').read_text())).set_index('threshold')
       for name in ('wildreceipt', 'cord')}
pd.concat({'WildReceipt (should be ~0)': cal['wildreceipt'].test_replication,
           'CORD official test': cal['cord'].test_replication}, axis=1).style.format('{:.1%}')

## 2. Extraction results (entity F1, mean ± std over seeds)

In [ ]:
!python -m docext.summarise runs --split test

## 3. Routing: share of documents that can skip review

Documents are ranked by a label-free risk signal; the safest share is auto-processed. A better signal keeps F1 high while covering more documents. *Oracle* ranks by true errors and is the upper bound.

In [ ]:
display(Image(f'{RUN}/routing_test/risk_coverage.png'))
pd.DataFrame(json.loads(Path(f'{RUN}/routing_test/summary.json').read_text())['signals']).T.round(3)

## 4. One document, end to end
Green: correct field · red: wrong · amber: missed.

In [ ]:
risky = pick(RUN, 'test', 'riskiest', 3)
print('riskiest documents:', risky)
show(RUN, 'test', risky[0])

In [ ]:
safe = pick(RUN, 'test', 'safest', 3)
show(RUN, 'test', safe[0])

## 5. How the experiments ran
Every experiment is a job in `jobs.json`; a runner on a GPU executed them in dependency order and published each result to the `results` branch.

In [ ]:
status = json.loads(Path('status.json').read_text())
jobs = json.loads(Path('jobs.json').read_text())['jobs']
print(f"{len(jobs)} jobs; " + ", ".join(f"{s}: {sum(v['state'] == s for v in status.values())}" for s in ('done', 'running', 'failed')))
pd.DataFrame(status).T[['state', 'minutes']].tail(10)